# Tool calling with LangGraph + Azure OpenAI — how Claude-style "data analysis" works

When you drop a CSV, Excel or text file into an assistant and ask a question, the model **never reads the file itself**. Instead:

1. It is given **tools** (here: `list_files` and `run_python`).
2. It **writes Python code** (pandas, `open()`, regex...) and asks us to run it via a *tool call*.
3. We **execute** the code, and send stdout / errors back as the *tool result*.
4. The model reads the result, writes more code if needed (including fixing its own errors), and finally answers in plain English.

This notebook builds that loop **step by step** with a LangGraph `StateGraph` and the plain `openai` `AzureOpenAI` client (no LangChain), on sample **trades** and **accounts** data:

| File | Type | Content |
|---|---|---|
| `trades.csv` | CSV | ~200 trades (with a few deliberate data problems) |
| `accounts.xlsx` | Excel | `Accounts` sheet + `Limits` sheet (with a title row above the header) |
| `ops_notes.txt` | Text | free-text trade-desk / ops log |

**Steps:** 1 Install · 2 Configure · 3 Generate sample data · 4 Define the tools · 5 Test the tools by hand · 6 Build the LangGraph · 7 Ask questions (CSV, Excel, text, cross-file, limits) · 8 Update records dynamically with rules (new columns, saved to `output/`) · 9 Caveats.

The API key is read from `AZURE_OPENAI_KEY` (or prompted) — it is never stored in the notebook.

## Step 1 — Install dependencies

In [ ]:
"""Install LangGraph (the graph runtime), the OpenAI SDK (AzureOpenAI client), pandas and openpyxl (Excel)."""
%pip install --quiet langgraph openai pandas openpyxl

## Step 2 — Configuration

Set `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT` and `AZURE_OPENAI_KEY` as environment variables (or edit the placeholders below). We use the **Responses API** (`client.responses.create`), which supports function tools.

In [ ]:
"""Azure OpenAI client + deployment, data folder, and loop limits.

    - OPENAI_ENDPOINT / OPENAI_API_VERSION / MODEL: your Azure OpenAI resource and deployment (env vars AZURE_OPENAI_ENDPOINT / AZURE_OPENAI_DEPLOYMENT).
    - OPENAI_KEY: from the AZURE_OPENAI_KEY env var, else prompted via getpass (never stored in the notebook).
    - DATA_DIR: where the sample files are written and where model-generated code runs.
    - MAX_STEPS: safety cap on model turns per question (stops runaway tool loops).
    - CODE_TIMEOUT_S / MAX_OUTPUT_CHARS: limits for each run_python call.
"""
import os, getpass
from pathlib import Path
from openai import AzureOpenAI

OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "https://<your-resource>.cognitiveservices.azure.com/")
OPENAI_API_VERSION = "2025-04-01-preview"
MODEL = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "<your-deployment-name>")   # Azure deployment name (must support function tools)

OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY", "").strip() or getpass.getpass("Azure OpenAI key: ").strip()
client = AzureOpenAI(azure_endpoint=OPENAI_ENDPOINT, api_version=OPENAI_API_VERSION, api_key=OPENAI_KEY)

DATA_DIR = Path.cwd() / "sample_data"
DATA_DIR.mkdir(exist_ok=True)
MAX_STEPS = 8
CODE_TIMEOUT_S = 30
MAX_OUTPUT_CHARS = 4000
print("Model :", MODEL)
print("Data  :", DATA_DIR)

## Step 3 — Generate the sample data

Three linked files (they share `account_id`), seeded so every run is identical. Deliberate quirks make the model earn its answer:

- `trades.csv`: a few missing venues, one **duplicate** `trade_id`, some `FAILED` / `PENDING` trades.
- `accounts.xlsx`: the `Limits` sheet has a **title row above the real header**, so a naive `pd.read_excel` gives the wrong columns.
- `ops_notes.txt`: free text that mentions real failed trade IDs and their reasons.

In [ ]:
"""Write sample_data/trades.csv, sample_data/accounts.xlsx and sample_data/ops_notes.txt (deterministic, seed=7)."""
import random
import pandas as pd

random.seed(7)

# ---- accounts.xlsx : Accounts sheet -------------------------------------
acct_rows = [
    ("ACC-101", "Northwind Pension Fund",   "Fund",          "EMEA", "A. Keller"),
    ("ACC-102", "Helios Capital",           "Institutional", "EMEA", "A. Keller"),
    ("ACC-103", "J. Whitaker",              "Retail",        "AMER", "R. Moreno"),
    ("ACC-104", "Orchid Asset Mgmt",        "Institutional", "APAC", "L. Tan"),
    ("ACC-105", "Baltic Insurance Group",   "Institutional", "EMEA", "A. Keller"),
    ("ACC-106", "M. Okafor",                "Retail",        "EMEA", "S. Brandt"),
    ("ACC-107", "Pacific Growth Fund",      "Fund",          "APAC", "L. Tan"),
    ("ACC-108", "Summit Endowment",         "Fund",          "AMER", "R. Moreno"),
    ("ACC-109", "P. Nair",                  "Retail",        "APAC", "L. Tan"),
    ("ACC-110", "Atlas Sovereign Wealth",   "Institutional", "EMEA", "S. Brandt"),
    ("ACC-111", "C. Dubois",                "Retail",        "EMEA", "S. Brandt"),
    ("ACC-112", "Redwood Hedge Partners",   "Fund",          "AMER", "R. Moreno"),
]
accounts = pd.DataFrame(acct_rows, columns=["account_id", "account_name", "type", "region", "relationship_manager"])
accounts["open_date"] = [f"20{random.randint(15, 23)}-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}" for _ in acct_rows]

# ---- accounts.xlsx : Limits sheet (title rows above the header!) --------
limit_by_type = {"Retail": 25_000, "Fund": 750_000, "Institutional": 2_000_000}
limits = pd.DataFrame({
    "account_id": accounts["account_id"],
    "max_trade_notional": accounts["type"].map(limit_by_type),
    "daily_limit": accounts["type"].map(limit_by_type) * 3,
})
xlsx_path = DATA_DIR / "accounts.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    accounts.to_excel(xw, sheet_name="Accounts", index=False)
    limits.to_excel(xw, sheet_name="Limits", index=False, startrow=3)
    ws = xw.sheets["Limits"]
    ws["A1"] = "Account Limits - CONFIDENTIAL"
    ws["A2"] = "Source: Risk team, updated 2025-09-30"

# ---- trades.csv ---------------------------------------------------------
base_price = {"AAPL": 225, "MSFT": 420, "NVDA": 120, "TSLA": 250, "SAP": 190, "HSBA": 7, "TM": 180, "BABA": 90}
venues = ["NYSE", "NASDAQ", "LSE", "XETRA", "TSE"]
trades = []
for i in range(200):
    sym = random.choice(list(base_price))
    trades.append({
        "trade_id": f"T{1001 + i}",
        "trade_date": f"2025-09-{random.randint(1, 30):02d}",
        "account_id": random.choice(accounts["account_id"].tolist()),
        "symbol": sym,
        "side": random.choice(["BUY", "SELL"]),
        "quantity": random.choice([50, 100, 250, 500, 1000, 2500, 5000]),
        "price": round(base_price[sym] * random.uniform(0.95, 1.05), 2),
        "currency": "GBP" if sym == "HSBA" else "USD",
        "venue": random.choice(venues),
        "status": random.choices(["SETTLED", "PENDING", "FAILED"], weights=[88, 4, 8])[0],
    })
trades = pd.DataFrame(trades)
for idx in random.sample(range(len(trades)), 5):
    trades.loc[idx, "venue"] = None                      # missing venues
trades = pd.concat([trades, trades.iloc[[10]]], ignore_index=True)   # one duplicated trade_id
trades.to_csv(DATA_DIR / "trades.csv", index=False)

# ---- ops_notes.txt (mentions real failed / pending trades) --------------
reasons = ["SSI mismatch, chasing custodian", "insufficient cash in settlement account", "counterparty did not affirm",
           "wrong ISIN on confirmation, amended and resubmitted", "CSD cut-off missed"]
lines = []
for _, t in trades[trades.status == "FAILED"].drop_duplicates("trade_id").iterrows():
    lines.append(f"{t.trade_date} {random.randint(8, 17):02d}:{random.randint(0, 59):02d} | OPS | {t.trade_id} ({t.account_id}) failed settlement: {random.choice(reasons)}.")
for _, t in trades[trades.status == "PENDING"].drop_duplicates("trade_id").iterrows():
    lines.append(f"{t.trade_date} {random.randint(8, 17):02d}:{random.randint(0, 59):02d} | OPS | {t.trade_id} ({t.account_id}) still pending, awaiting counterparty confirmation.")
lines += [
    "2025-09-03 10:12 | DESK | ACC-103 asked why the AAPL fill price differed from the quote shown. Explained spread and slippage.",
    "2025-09-17 14:30 | RISK | Reminder: Retail accounts have a per-trade notional limit, see the Limits sheet in accounts.xlsx.",
    "2025-09-22 09:05 | DESK | ACC-110 requested a call about their NVDA allocation. Scheduled with A. Keller.",
    "2025-09-29 16:45 | COMPLIANCE | Reviewing ACC-112 for possible concentration in TSLA. No action yet.",
]
lines.sort()
(DATA_DIR / "ops_notes.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

for p in sorted(DATA_DIR.iterdir()):
    print(f"{p.name:18} {p.stat().st_size:>7,} bytes")
print("\nstatus counts:", trades.status.value_counts().to_dict())

## Step 4 — Define the tools

A tool is two things: a **JSON schema** the model sees (name, description, parameters), and a **real Python function** we run when the model asks for it. The model only ever produces the *name* and the *arguments*; it cannot execute anything itself.

`run_python` is the important one: it takes the model's code, runs it in a **separate Python process** inside `DATA_DIR` (so relative paths like `trades.csv` work), and returns whatever was printed — or the traceback if it crashed.

In [ ]:
"""Tool implementations + the tool schemas sent to the model.

    list_files()       -> JSON list of the files in DATA_DIR (name, size, type).
    run_python(code)   -> runs the code in a fresh `python` subprocess (cwd=DATA_DIR, timeout), returns stdout + stderr.
                          Output is truncated to MAX_OUTPUT_CHARS so one huge print can't flood the context window.
    TOOLS              -> function-tool schemas in the Responses API format.
    run_tool(name, args) -> dispatcher used by the graph's tools node.
"""
import json, subprocess, sys, tempfile

def list_files() -> str:
    files = [{"name": p.name, "bytes": p.stat().st_size, "type": p.suffix.lstrip(".")} for p in sorted(DATA_DIR.iterdir()) if p.is_file()]
    return json.dumps(files)

def run_python(code: str) -> str:
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, dir=tempfile.gettempdir(), encoding="utf-8") as f:
        f.write(code)
        script = f.name
    try:
        proc = subprocess.run([sys.executable, script], cwd=DATA_DIR, capture_output=True, text=True, timeout=CODE_TIMEOUT_S)
        out = proc.stdout + (f"\n[stderr]\n{proc.stderr}" if proc.stderr else "")
    except subprocess.TimeoutExpired:
        out = f"[error] code timed out after {CODE_TIMEOUT_S}s"
    finally:
        os.unlink(script)
    out = out.strip() or "[no output - did you forget to print()?]"
    return out if len(out) <= MAX_OUTPUT_CHARS else out[:MAX_OUTPUT_CHARS] + f"\n...[truncated {len(out) - MAX_OUTPUT_CHARS} chars]"

TOOLS = [
    {
        "type": "function",
        "name": "list_files",
        "description": "List the data files available to analyse (name, size in bytes, file type).",
        "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False},
    },
    {
        "type": "function",
        "name": "run_python",
        "description": ("Run Python 3 code (pandas and openpyxl are installed) in the data folder and return stdout/stderr. "
                        "Files are read with relative paths, e.g. pd.read_csv('trades.csv'). Nothing persists between calls, "
                        "so each call must import and load what it needs. Use print() to see results."),
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "Complete, self-contained Python code to execute."}},
            "required": ["code"],
            "additionalProperties": False,
        },
    },
]

def run_tool(name: str, args: dict) -> str:
    if name == "list_files":
        return list_files()
    if name == "run_python":
        return run_python(args["code"])
    return f"[error] unknown tool {name!r}"

## Step 5 — Test the tools by hand (no LLM yet)

This is exactly what the model will do for us, except that *we* write the code. Note the third call: a deliberately naive read of the `Limits` sheet. Look at the columns it returns — this is the trap the model can fall into later, and odd output or a traceback is what it learns from.

In [ ]:
"""Call the tools directly: list the files, peek at the CSV, read the Excel 'Limits' sheet naively, then trigger an error."""
print(run_tool("list_files", {}))
print("-" * 60)
print(run_tool("run_python", {"code": "import pandas as pd\ndf = pd.read_csv('trades.csv')\nprint(df.head(3).to_string())\nprint(df.shape)"}))
print("-" * 60)
print(run_tool("run_python", {"code": "import pandas as pd\nprint(pd.read_excel('accounts.xlsx', sheet_name='Limits').head(3).to_string())"}))
print("-" * 60)
print(run_tool("run_python", {"code": "print(undefined_variable)"}))   # an error comes back as text, not an exception

## Step 6 — Build the LangGraph

The whole "agent" is a loop of two nodes:

```
START -> agent --(model asked for tools)--> tools --> agent --> ... --> END
              \--(plain answer)---------------------------------------> END
```

- **`agent` node**: calls the model. The model returns either text (done) or one or more `function_call` items.
- **`tools` node**: runs each requested call and returns a `function_call_output` item with the result.
- **Conditional edge**: if there are pending tool calls (and we are under `MAX_STEPS`) go to `tools`, otherwise stop.

The Responses API stores the conversation server-side, so each turn we only send the *new* items plus `previous_response_id`. The system prompt and tool schemas are sent every turn.

The state holds: the next input items, the last response id, the pending tool calls, a step counter and the final answer.

In [ ]:
"""LangGraph state, system prompt, the two nodes, the router, and the compiled graph (`app`).

    State.input     : items to send on the next model call (user question, then tool outputs).
    State.prev_id   : id of the previous response, so the server keeps the history for us.
    State.pending   : function_call items the model requested in its last turn.
    State.steps     : model turns used so far (compared with MAX_STEPS).
    State.answer    : the model's final plain-text answer.
    State.system    : system prompt for this run (lets the same graph answer questions or update records).
"""
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

SYSTEM_PROMPT = """You are a data analyst for a trading operations team. The data files are:
trades.csv (trades), accounts.xlsx (sheets: Accounts and Limits) and ops_notes.txt (ops log). account_id links them.

Rules:
- Never guess. Use list_files, then inspect each file you need (columns, sheet names, header rows, a few sample rows) BEFORE doing the analysis.
- Do the analysis by writing Python and running it with run_python. Always print() what you need to see.
- If the code errors or the data looks odd (duplicates, nulls, title rows), investigate and fix it, and mention it in your answer.
- Finish with a short, plain-English answer that states the numbers you computed."""

class State(TypedDict):
    input: list
    prev_id: Optional[str]
    pending: list
    steps: int
    answer: str
    system: str

def agent_node(state: State) -> dict:
    """Call the model with the new input items; return its tool calls (if any) or its final text."""
    resp = client.responses.create(
        model=MODEL,
        instructions=state["system"],
        input=state["input"],
        previous_response_id=state["prev_id"],
        tools=TOOLS,
        reasoning={"effort": "low"},
        max_output_tokens=8000,
    )
    calls = [it.model_dump(exclude_none=True) for it in resp.output if it.type == "function_call"]
    return {"prev_id": resp.id, "pending": calls, "steps": state["steps"] + 1, "answer": "" if calls else resp.output_text, "input": []}

def tools_node(state: State) -> dict:
    """Execute every pending tool call and package the results as function_call_output items."""
    outputs = []
    for call in state["pending"]:
        result = run_tool(call["name"], json.loads(call["arguments"] or "{}"))
        outputs.append({"type": "function_call_output", "call_id": call["call_id"], "output": result})
    return {"input": outputs, "pending": []}

def route(state: State) -> str:
    """Go to the tools node while the model keeps asking for tools, but never beyond MAX_STEPS."""
    if state["pending"] and state["steps"] < MAX_STEPS:
        return "tools"
    return END

graph = StateGraph(State)
graph.add_node("agent", agent_node)
graph.add_node("tools", tools_node)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", route, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")
app = graph.compile()
print("graph compiled")

In [ ]:
"""Show the compiled graph as Mermaid source and (in JupyterLab 4+) as a rendered diagram."""
from IPython.display import display, Markdown
mermaid = app.get_graph().draw_mermaid()
print(mermaid)
display(Markdown("```mermaid\n" + mermaid + "\n```"))

### The `ask()` helper — watch the loop run

`ask(question)` streams the graph one node at a time and prints each step:

- **model turn** — what the model decided (which tool, and the **Python it wrote**)
- **tool result** — what our code printed back
- **answer** — the final plain-English reply

In [ ]:
"""ask(question): run the graph and print every model turn, generated code and tool result as it happens.

    Returns the final answer text. Set show_code=False to hide the generated code; pass system=... to change the instructions.
"""
def ask(question: str, show_code: bool = True, system: str = SYSTEM_PROMPT) -> str:
    state = {"input": [{"role": "user", "content": question}], "prev_id": None, "pending": [], "steps": 0, "answer": "", "system": system}
    print("QUESTION:", question, "\n")
    answer = ""
    for update in app.stream(state, stream_mode="updates"):
        for node, delta in update.items():
            if node == "agent":
                if delta["pending"]:
                    for call in delta["pending"]:
                        args = json.loads(call["arguments"] or "{}")
                        print(f"[model turn {delta['steps']}] -> tool call: {call['name']}")
                        if show_code and "code" in args:
                            print("    " + args["code"].replace("\n", "\n    "))
                else:
                    answer = delta["answer"]
            elif node == "tools":
                for out in delta["input"]:
                    text = out["output"]
                    print("[tool result]")
                    print("    " + (text if len(text) < 1500 else text[:1500] + "\n    ...").replace("\n", "\n    "))
                print()
    print("=" * 80)
    print("ANSWER:\n" + (answer or "[no final answer - hit MAX_STEPS?]"))
    return answer

## Step 7 — Ask questions

### 7a. CSV — `trades.csv`
The model should inspect the columns first, notice the duplicate `trade_id`, and then aggregate.

In [ ]:
_ = ask("What are the top 3 symbols by total traded notional (quantity * price)? Are there any data quality problems in trades.csv I should know about?")

### 7b. Excel — `accounts.xlsx`
The model has to discover the sheet names, and then handle the title rows above the `Limits` header (the trap from Step 5). Watch for an odd first attempt followed by a corrected one — that is the **self-correction** loop.

In [ ]:
_ = ask("List the Institutional accounts and their max_trade_notional from the Limits sheet. Also tell me how many sheets accounts.xlsx has and what they are.")

### 7c. Text — `ops_notes.txt`
No pandas needed: the model reads the file and uses string handling / regex to pull structured facts out of free text.

In [ ]:
_ = ask("From ops_notes.txt, list every trade ID that failed settlement together with the reason given, and count how many failed for each reason.")

### 7d. Cross-file — CSV + Excel + text together
The model has to plan a multi-step job: read trades, join to account details, find the failures, then correlate with the notes. Several tool calls, each building on the last.

In [ ]:
_ = ask("For Institutional accounts in EMEA, which trades failed settlement, and what do the ops notes say about each one?")

### 7e. Limit breaches — join + business logic
Needs the (title-row) `Limits` sheet joined to the trades: which trades exceeded the account's `max_trade_notional`?

In [ ]:
_ = ask("Which trades breached their account's max_trade_notional? Show a count per account type and list the 5 biggest breaches with account name and notional.")

### 7f. Your turn
Change the question below. Try asking something the data cannot answer (e.g. a column that does not exist) and see whether the model says so instead of inventing a number.

In [ ]:
_ = ask("Which relationship manager has the most failed trades across their accounts, and what was the total notional of those failed trades?")

## Step 8 — Update records dynamically with rules

So far the model only *read* data. The same loop can **change** it: you state a rule in plain English ("price above 200 is Bullish"), the model turns it into pandas code, runs it, and writes the result.

Guardrails we add for write jobs:
- **Originals are never overwritten.** The model must save to `sample_data/output/`.
- The model must **reload the saved file and print a summary** to prove the change landed.
- Afterwards *we* re-check the result with our own independent pandas code, so we are not just trusting the model's word.

The graph is unchanged — only the system prompt differs (`ask(..., system=UPDATE_PROMPT)`).

In [ ]:
"""UPDATE_PROMPT (write-job instructions) and OUT_DIR (where updated copies are saved).

    The model gets the usual analyst rules plus: never modify originals, save to output/, preserve everything not being changed,
    reload and summarise the saved file, and report the exact rule applied.
"""
assert (DATA_DIR / "trades.csv").exists(), "Sample data is missing - re-run Step 3 first"
OUT_DIR = DATA_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

UPDATE_PROMPT = SYSTEM_PROMPT + """

You are now doing UPDATE jobs: the user gives a business rule and you apply it to the data.
- NEVER modify or overwrite the original files. Save the updated copy under output/ using the file name the user gives.
- Keep every existing row, column, sheet and line unchanged except for what the rule requires (for Excel keep all sheets, and keep any title rows above a header).
- Apply the rule exactly as stated, with a single vectorised pandas expression where possible (e.g. numpy.select), and keep the rows that match no rule explicit (use the label the user gives).
- After saving, reload the saved file and print: the row count, the new column's value counts and 5 sample rows.
- In your final answer state the exact rule you applied and how many records fell into each bucket."""
print("Updated copies will be saved to:", OUT_DIR)

### 8a. CSV — add a computed label column
The rule lives in a plain Python string, so you can change the thresholds or labels and re-run to get a different result with **no code changes** — the model re-writes the code each time.

In [ ]:
RULE = "price above 200 is 'Bullish', price below 100 is 'Bearish', everything else is 'Neutral'"

_ = ask(f"In trades.csv add a new column named 'outlook' using this rule: {RULE}. Save the result as output/trades_enriched.csv.", system=UPDATE_PROMPT)

In [ ]:
"""Independent check of 8a: recompute the rule ourselves and compare with what the model wrote."""
import numpy as np
orig = pd.read_csv(DATA_DIR / "trades.csv")
new = pd.read_csv(OUT_DIR / "trades_enriched.csv")
expected = np.select([orig.price > 200, orig.price < 100], ["Bullish", "Bearish"], default="Neutral")

print("same row count             :", len(new) == len(orig))
print("original columns untouched :", new[orig.columns].equals(orig))
print("new columns                :", [c for c in new.columns if c not in orig.columns])
print("outlook matches our rule   :", (new["outlook"].to_numpy() == expected).all())
print(new["outlook"].value_counts().to_string())
new.head()

### 8b. Excel — update one sheet without damaging the workbook
Harder: the `Limits` sheet has title rows, and `Accounts` must survive untouched. A careless `df.to_excel()` would drop a sheet or lose the title rows; the update prompt tells the model to preserve both.

In [ ]:
_ = ask("In accounts.xlsx, on the Limits sheet add a column 'size_flag': 'HIGH' if max_trade_notional is at least 1,000,000, 'MEDIUM' if at least 500,000, otherwise 'LOW'. Keep the Accounts sheet and the title rows on Limits. Save as output/accounts_updated.xlsx.", system=UPDATE_PROMPT)

In [ ]:
"""Check 8b: both sheets still exist, title rows kept, new column present and correct."""
xl = pd.ExcelFile(OUT_DIR / "accounts_updated.xlsx")
print("sheets:", xl.sheet_names)
raw = pd.read_excel(xl, sheet_name="Limits", header=None)
print("title row kept:", raw.iloc[0, 0])
lim = pd.read_excel(xl, sheet_name="Limits", skiprows=3)
exp = np.select([lim.max_trade_notional >= 1_000_000, lim.max_trade_notional >= 500_000], ["HIGH", "MEDIUM"], default="LOW")
print("size_flag matches our rule:", (lim["size_flag"].to_numpy() == exp).all())
lim

### 8c. Text — tag every line of the notes
Same idea for unstructured text: the "record" is a line, the rule is keyword-based, and the output is a tagged copy.

In [ ]:
_ = ask("Copy ops_notes.txt to output/ops_notes_tagged.txt, prefixing each line with [FAILED] if it mentions 'failed settlement', [PENDING] if it mentions 'pending', otherwise [INFO]. Report how many lines got each tag.", system=UPDATE_PROMPT)
print((OUT_DIR / "ops_notes_tagged.txt").read_text(encoding="utf-8")[:900])

### 8d. Multi-rule update with a join
Combine data from two files: flag trades whose notional exceeds the account's `max_trade_notional` **and** apply the outlook rule at the same time. The model must work out the join and the title-row handling by itself.

In [ ]:
_ = ask(f"In trades.csv add two columns: 'outlook' ({RULE}) and 'limit_breach' (True if quantity * price exceeds that account's max_trade_notional from the Limits sheet of accounts.xlsx, else False). Save as output/trades_flagged.csv and tell me how many trades breach, per account type.", system=UPDATE_PROMPT)

## Step 9 — What just happened, and caveats

**The protocol, in one picture**

```
user question
   -> model: "call run_python with this code"   (a function_call item: name + JSON arguments)
   -> we run it, capture stdout/stderr          (a function_call_output item)
   -> model reads the output, writes more code, or answers
```

**Things to notice**
- The model *explored before computing*: `list_files`, then columns / sheet names / sample rows. The system prompt asks for this, but the same habit is why Claude-style assistants rarely guess a schema.
- **Errors are just more input.** A traceback goes back as the tool result and the model fixes its code next turn.
- **The model's numbers are only as good as the code it ran** — the printed output is the evidence. Always read the trace, not only the answer.
- **Writes are the same loop as reads.** Step 8 only changed the system prompt: the model turned a plain-English rule into pandas code and saved a new file. Verify with your own code (as in 8a/8b), and keep originals untouched.
- Each `run_python` call is a fresh process, so state does not persist. Real products keep a persistent kernel instead.

**Safety — important**
- `run_python` executes **model-written code with your user's permissions** (files, network, environment variables, including your API key). That is fine for a learning notebook with your own sample data. Write jobs raise the stakes (a bad rule could overwrite data), so here originals are protected only by instruction; a real system should enforce it with read-only mounts. Never point this at untrusted files: a text file can contain instructions that make the model write harmful code (*prompt injection*).
- For production use a real sandbox: a container / microVM with no network, a read-only data mount, CPU/memory limits and a short timeout (e.g. Azure Container Apps dynamic sessions).

**Next ideas:** keep a persistent Jupyter kernel between calls, add a `write_file`/chart tool, add a human-approval node before code runs (LangGraph `interrupt`), add a human-approval node that shows the proposed change before it is saved, or give it a SQL tool next to `run_python`.